# SIA — Train on Colab, then upload to Hugging Face

Run this in **Google Colab with a GPU**: Runtime → Change runtime type → **T4 GPU**.

Order: upload code → install → upload dataset → train → check → upload model to Hugging Face.
After this finishes, deploy the app pointing at your Hugging Face model repo.

This notebook trains **`deberta-v3-xsmall`** — a small, memory-light model chosen so the
app fits free hosting (e.g. Streamlit Community Cloud's ~1 GB limit). If you have more
memory you can switch `--model` to `microsoft/deberta-v3-small` or `-base`.


### 1. Upload the project code
Run the cell, then pick your `support-integrity-auditor.zip`.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()
z = [f for f in up if f.endswith('.zip')][0]
with zipfile.ZipFile(z) as f:
    f.extractall('.')
os.chdir('support-integrity-auditor')
print('now in:', os.getcwd())
print(os.listdir())

### 2. Install the training dependencies
Colab already has PyTorch with GPU, so we do **not** reinstall torch — just the rest.

In [ ]:
!pip -q install transformers==4.44.2 accelerate==0.33.0 sentencepiece==0.2.0 protobuf==5.27.2 huggingface_hub==0.24.6

### 3. Upload the dataset
Download the **Customer Support Tickets — CRM Dataset** from Kaggle
(kaggle.com/datasets/ajverse/customersupport-tickets-crm-dataset), then run this cell and select the CSV.

In [ ]:
from google.colab import files
import os, pandas as pd
os.makedirs('data', exist_ok=True)
up = files.upload()
csv = [f for f in up if f.endswith('.csv')][0]
os.replace(csv, 'data/customer_support_tickets.csv')
df = pd.read_csv('data/customer_support_tickets.csv')
print('rows, cols:', df.shape)
print('COLUMNS:', list(df.columns))   # <-- check these match what the pipeline expects

**Check the columns above.** The pipeline reads `Ticket Subject`, `Ticket Description`,
`Ticket Priority`, `Ticket Channel`, and — for the resolution-time signal — `First Response Time`
and `Time to Resolution`. If your file instead has a single `Resolution Time` column (or different
names), the resolution signal will quietly produce nothing, leaving only one signal (the brief
requires >= 2). If so, send the real column names and adjust `src/signals.py` before training.

### 4. Train (deberta-v3-xsmall)
A few minutes on a T4. Prints the metrics and a `verified: true/false` flag at the end.
xsmall is small, so we give it an extra epoch; bump `--epochs` further if it lands just short.

In [ ]:
!python train_pipeline.py --csv data/customer_support_tickets.csv --out artifacts --model microsoft/deberta-v3-xsmall --epochs 5

### 5. Confirm it passed the thresholds
Need accuracy >= 0.83, macro_f1 >= 0.82, both recalls >= 0.78.

In [ ]:
import json
print(json.dumps(json.load(open('artifacts/metrics.json')), indent=2))

### 6. Upload the trained model to Hugging Face
Get a **write** token at huggingface.co/settings/tokens. Change the repo name to your username.

In [ ]:
from huggingface_hub import login, HfApi
login()  # paste your HF write token

REPO = "YOUR-HF-USERNAME/sia-deberta"   # <-- CHANGE THIS

api = HfApi()
api.create_repo(REPO, repo_type="model", exist_ok=True)
api.upload_folder(folder_path="artifacts/model", repo_id=REPO)
api.upload_file(path_or_fileobj="artifacts/calibration.json",
                path_in_repo="calibration.json", repo_id=REPO)
print("Uploaded to https://huggingface.co/" + REPO)

### 7. Deploy
Set these two values on your host (Streamlit Cloud → Secrets, or HF Spaces → Variables):

```
SIA_MODEL = YOUR-HF-USERNAME/sia-deberta
SIA_CALIBRATION = calibration.json
```

**Streamlit Community Cloud reminder:** create the app with **Python 3.11** in Advanced settings
(it cannot be changed later). That URL is your submission.